In [1]:
import pandas as pd
print("working")

ModuleNotFoundError: No module named 'pandas'

In [2]:
%pip install pandas

                                              0.0/11.0 MB ? eta -:--:--
                                              0.0/11.0 MB ? eta -:--:--
                                              0.0/11.0 MB 1.9 MB/s eta 0:00:06
                                              0.0/11.0 MB 1.9 MB/s eta 0:00:06
                                             0.1/11.0 MB 544.7 kB/s eta 0:00:21
                                             0.1/11.0 MB 901.1 kB/s eta 0:00:13
                                             0.1/11.0 MB 901.1 kB/s eta 0:00:13
                                             0.2/11.0 MB 831.5 kB/s eta 0:00:13
                                             0.2/11.0 MB 831.5 kB/s eta 0:00:13
                                             0.2/11.0 MB 654.9 kB/s eta 0:00:17
                                             0.3/11.0 MB 741.6 kB/s eta 0:00:15
     -                                       0.3/11.0 MB 679.5 kB/s eta 0:00:16
     -                                       0.3/11.0 MB 6

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
print("working")

working


In [2]:
import pandas as pd

path = "../data/raw/flights.csv"
parts = []

for chunk in pd.read_csv(path, chunksize=200_000, low_memory=False):
    chunk = chunk[chunk["MONTH"].isin([1, 2, 3])]
    parts.append(chunk)

flights = pd.concat(parts, ignore_index=True)
flights = flights.sample(n=400_000, random_state=42)
flights.to_csv("../data/raw/flights_subset.csv", index=False)

print(flights.shape)

(400000, 31)


In [3]:
flights.info()
print(flights.isnull().sum())
print("Cancelled %:", flights["CANCELLED"].mean() * 100)
print("Diverted %:", flights["DIVERTED"].mean() * 100)
print(flights["ORIGIN_AIRPORT"].astype(str).str.len().value_counts())

<class 'pandas.DataFrame'>
Index: 400000 entries, 610909 to 985296
Data columns (total 31 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   YEAR                 400000 non-null  int64  
 1   MONTH                400000 non-null  int64  
 2   DAY                  400000 non-null  int64  
 3   DAY_OF_WEEK          400000 non-null  int64  
 4   AIRLINE              400000 non-null  str    
 5   FLIGHT_NUMBER        400000 non-null  int64  
 6   TAIL_NUMBER          397693 non-null  str    
 7   ORIGIN_AIRPORT       400000 non-null  object 
 8   DESTINATION_AIRPORT  400000 non-null  object 
 9   SCHEDULED_DEPARTURE  400000 non-null  int64  
 10  DEPARTURE_TIME       387842 non-null  float64
 11  DEPARTURE_DELAY      387842 non-null  float64
 12  TAXI_OUT             387620 non-null  float64
 13  WHEELS_OFF           387620 non-null  float64
 14  SCHEDULED_TIME       399999 non-null  float64
 15  ELAPSED_TIME         386695 

In [4]:
airports = pd.read_csv("../data/raw/airports.csv")

print(flights[["ARRIVAL_DELAY", "CANCELLED", "CANCELLATION_REASON"]].isnull().sum())
print("Cancelled %:", flights["CANCELLED"].mean() * 100)
print(flights["ARRIVAL_DELAY"].describe())
print(flights["MONTH"].value_counts().sort_index())
print("Origins not in airports.csv:",
      set(flights["ORIGIN_AIRPORT"]) - set(airports["IATA_CODE"]))

ARRIVAL_DELAY           13305
CANCELLED                   0
CANCELLATION_REASON    387527
dtype: int64
Cancelled %: 3.1182499999999997
count    386695.000000
mean          6.260911
std          40.522467
min         -74.000000
25%         -12.000000
50%          -3.000000
75%          10.000000
max        1593.000000
Name: ARRIVAL_DELAY, dtype: float64
MONTH
1    133602
2    122434
3    143964
Name: count, dtype: int64
Origins not in airports.csv: set()


In [5]:
mask = (flights["CANCELLED"] == 1) | (flights["DIVERTED"] == 1)
print(mask.sum(), flights.loc[mask, "ARRIVAL_DELAY"].isnull().sum())

13305 13305


In [6]:
flights["FlightDate"] = pd.to_datetime(flights[["YEAR", "MONTH", "DAY"]])
flights[["FlightDate", "YEAR", "MONTH", "DAY"]].head()

,FlightDate,YEAR,MONTH,DAY
610909,2015-02-10,2015,2,10
167314,2015-01-11,2015,1,11
1101774,2015-03-13,2015,3,13
809238,2015-02-23,2015,2,23
613237,2015-02-10,2015,2,10


In [7]:
flights["IsDelayed"] = (flights["ARRIVAL_DELAY"] >= 15).astype("Int64")
flights.loc[flights["ARRIVAL_DELAY"].isnull(), "IsDelayed"] = pd.NA

flights["IsDelayed"].value_counts(dropna=False)

IsDelayed
0       305102
1        81593
<NA>     13305
Name: count, dtype: Int64

In [8]:
bins   = [-float("inf"), 0, 15, 60, 120, float("inf")]
labels = ["Early/On time", "Minor (1-15)", "Moderate (16-60)", "Major (61-120)", "Severe (120+)"]

In [9]:
bins   = [-float("inf"), 0, 15, 60, 120, float("inf")]
labels = ["Early", "On time (0-14)", "Minor (15-59)", "Moderate (60-119)", "Severe (120+)"]

flights["DelayBucket"] = pd.cut(flights["ARRIVAL_DELAY"], bins=bins, labels=labels, right=False)

print(flights["DelayBucket"].value_counts(dropna=False))
print(flights.groupby("DelayBucket", observed=True)["IsDelayed"].agg(["min", "max"]))

DelayBucket
Early                221401
On time (0-14)        83701
Minor (15-59)         57180
Moderate (60-119)     16014
NaN                   13305
Severe (120+)          8399
Name: count, dtype: int64
                   min  max
DelayBucket                
Early                0    0
On time (0-14)       0    0
Minor (15-59)        1    1
Moderate (60-119)    1    1
Severe (120+)        1    1


In [10]:
reason_map = {"A": "Airline/Carrier", "B": "Weather", "C": "Air System (NAS)", "D": "Security"}
flights["CancellationReason"] = flights["CANCELLATION_REASON"].map(reason_map)

print(flights["CancellationReason"].value_counts(dropna=False))

CancellationReason
NaN                 387527
Weather               8445
Airline/Carrier       2313
Air System (NAS)      1712
Security                 3
Name: count, dtype: int64


In [11]:
cause_cols = ["AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY",
              "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY"]

print(flights[cause_cols].isnull().sum())
print(flights.loc[flights["IsDelayed"] == 1, cause_cols].isnull().sum())
print(flights.loc[flights["IsDelayed"] == 0, cause_cols].notnull().sum())

AIR_SYSTEM_DELAY       318407
SECURITY_DELAY         318407
AIRLINE_DELAY          318407
LATE_AIRCRAFT_DELAY    318407
WEATHER_DELAY          318407
dtype: int64
AIR_SYSTEM_DELAY       0
SECURITY_DELAY         0
AIRLINE_DELAY          0
LATE_AIRCRAFT_DELAY    0
WEATHER_DELAY          0
dtype: int64
AIR_SYSTEM_DELAY       0
SECURITY_DELAY         0
AIRLINE_DELAY          0
LATE_AIRCRAFT_DELAY    0
WEATHER_DELAY          0
dtype: int64


In [12]:
flights["ScheduledHour"] = flights["SCHEDULED_DEPARTURE"] // 100
print(flights["ScheduledHour"].value_counts().sort_index())

ScheduledHour
0       750
1       447
2        83
3        30
4        64
5      5934
6     28860
7     25928
8     27326
9     24294
10    25572
11    25345
12    23945
13    26802
14    23667
15    25249
16    24006
17    27800
18    22700
19    23008
20    17033
21    12410
22     6295
23     2452
Name: count, dtype: int64


In [13]:
flights.to_csv("../data/raw/flights_clean.csv", index=False)
print(flights.shape)
flights.dtypes

(400000, 36)


YEAR                            int64
MONTH                           int64
DAY                             int64
DAY_OF_WEEK                     int64
AIRLINE                           str
FLIGHT_NUMBER                   int64
TAIL_NUMBER                       str
ORIGIN_AIRPORT                 object
DESTINATION_AIRPORT            object
SCHEDULED_DEPARTURE             int64
DEPARTURE_TIME                float64
DEPARTURE_DELAY               float64
TAXI_OUT                      float64
WHEELS_OFF                    float64
SCHEDULED_TIME                float64
ELAPSED_TIME                  float64
AIR_TIME                      float64
DISTANCE                        int64
WHEELS_ON                     float64
TAXI_IN                       float64
SCHEDULED_ARRIVAL               int64
ARRIVAL_TIME                  float64
ARRIVAL_DELAY                 float64
DIVERTED                        int64
CANCELLED                       int64
CANCELLATION_REASON               str
AIR_SYSTEM_D

In [14]:
clean = pd.read_csv("../data/raw/flights_clean.csv", parse_dates=["FlightDate"])

print(clean.shape)
print(clean["FlightDate"].min(), clean["FlightDate"].max())
print("Max hour:", clean["ScheduledHour"].max())
print(clean[cause_cols].isnull().sum())
print(clean["CancellationReason"].value_counts(dropna=False))

(400000, 36)
2015-01-01 00:00:00 2015-03-31 00:00:00
Max hour: 23
AIR_SYSTEM_DELAY       318407
SECURITY_DELAY         318407
AIRLINE_DELAY          318407
LATE_AIRCRAFT_DELAY    318407
WEATHER_DELAY          318407
dtype: int64
CancellationReason
NaN                 387527
Weather               8445
Airline/Carrier       2313
Air System (NAS)      1712
Security                 3
Name: count, dtype: int64


In [15]:
print(clean.loc[clean["IsDelayed"] == 1, cause_cols].isnull().sum())   # sab 0 aane chahiye
print(clean.loc[clean["IsDelayed"] != 1, cause_cols].notnull().sum())  # sab 0 aane chahiye

AIR_SYSTEM_DELAY       0
SECURITY_DELAY         0
AIRLINE_DELAY          0
LATE_AIRCRAFT_DELAY    0
WEATHER_DELAY          0
dtype: int64
AIR_SYSTEM_DELAY       0
SECURITY_DELAY         0
AIRLINE_DELAY          0
LATE_AIRCRAFT_DELAY    0
WEATHER_DELAY          0
dtype: int64


In [16]:
import os

flights["CancellationReason"] = flights["CancellationReason"].fillna("Not Cancelled")

os.makedirs("../data/processed", exist_ok=True)
flights.to_csv("../data/processed/flights_clean.csv", index=False)

print(flights.shape)
print(flights["CancellationReason"].value_counts())

(400000, 36)
CancellationReason
Not Cancelled       387527
Weather               8445
Airline/Carrier       2313
Air System (NAS)      1712
Security                 3
Name: count, dtype: int64
